# PKR Vision - Kaggle registered classification driver
This notebook runs the same evidence-bound classification workflow on Kaggle. Attach the prepared dataset as a Kaggle Dataset, enable GPU, and keep outputs under `/kaggle/working`.

In [ ]:
REPOSITORY_URL = "REPLACE_WITH_PUBLIC_REPOSITORY_URL"
COMMIT = "7bd310f"
KAGGLE_DATASET_SLUG = "REPLACE_WITH_KAGGLE_DATASET_SLUG"
assert "REPLACE_WITH" not in REPOSITORY_URL
assert "REPLACE_WITH" not in COMMIT
assert "REPLACE_WITH" not in KAGGLE_DATASET_SLUG


In [ ]:
from pathlib import Path
import json
import platform
import shutil
import subprocess
import sys

WORKING = Path('/kaggle/working')
REPO_ROOT = WORKING / 'pkr-vision'
print(sys.version)
if sys.version_info < (3, 11) or sys.version_info >= (3, 14):
    raise RuntimeError('This project requires Python >=3.11,<3.14.')
if not shutil.which('nvidia-smi'):
    raise RuntimeError('No NVIDIA GPU detected. In Kaggle, open Notebook settings and enable GPU.')
subprocess.run(['nvidia-smi'], check=True)
if REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)
subprocess.run(['git', 'clone', '--filter=blob:none', REPOSITORY_URL, str(REPO_ROOT)], check=True)
subprocess.run(['git', 'checkout', COMMIT], cwd=REPO_ROOT, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[train]'], cwd=REPO_ROOT, check=True)
%cd /kaggle/working/pkr-vision
print(json.dumps({'python': sys.version, 'platform': platform.platform(), 'commit': COMMIT}, indent=2))


Attach the Kaggle Dataset that exposes `classification-v1`. Kaggle may unzip the archive automatically; the next cell prefers the extracted input path and falls back to `classification-v1.tar.gz` when present.


In [ ]:
from pathlib import Path
import hashlib
import json
import shutil
import tarfile

INPUT_ROOT = Path('/kaggle/input/datasets/hasnaatk/pkr-classification-v1')
EXTRACTED_INPUT = INPUT_ROOT / 'classification-v1'
ARCHIVE = INPUT_ROOT / 'classification-v1.tar.gz'
DATA_ROOT = Path('/kaggle/working/classification-v1')
EXPECTED_ARCHIVE = 'a6ccf9a55a43bbc638f25e4ef074dc775d7bf202c4ad2b3a341d85892c8d68ba'
EXPECTED_MANIFEST = 'd36533b491482a3eb4b84cdf6739ab346ac6f5d619fc59e088eca4d0e0e08413'

if EXTRACTED_INPUT.is_dir():
    source_root = EXTRACTED_INPUT
    print(f'Using extracted Kaggle input: {source_root}')
elif ARCHIVE.is_file():
    assert hashlib.sha256(ARCHIVE.read_bytes()).hexdigest() == EXPECTED_ARCHIVE
    if not DATA_ROOT.is_dir():
        with tarfile.open(ARCHIVE, 'r:gz') as bundle:
            bundle.extractall('/kaggle/working', filter='data')
    source_root = DATA_ROOT
    print(f'Extracted archive to: {source_root}')
else:
    raise FileNotFoundError(f'Expected either {EXTRACTED_INPUT} or {ARCHIVE}')

assert hashlib.sha256((source_root / 'samples.jsonl').read_bytes()).hexdigest() == EXPECTED_MANIFEST

target = Path('data/processed/classification-v1')
target.parent.mkdir(parents=True, exist_ok=True)
if target.exists() or target.is_symlink():
    if target.is_symlink() or target.is_file():
        target.unlink()
    else:
        shutil.rmtree(target)
target.symlink_to(source_root, target_is_directory=True)

RUN_ROOT = Path('/kaggle/working/artifacts/mlruns')
RUN_ROOT.mkdir(parents=True, exist_ok=True)
mlruns = Path('artifacts/mlruns')
mlruns.parent.mkdir(parents=True, exist_ok=True)
if mlruns.exists() or mlruns.is_symlink():
    if mlruns.is_symlink() or mlruns.is_file():
        mlruns.unlink()
    else:
        shutil.rmtree(mlruns)
mlruns.symlink_to(RUN_ROOT, target_is_directory=True)
print(json.dumps({'source_root': str(source_root), 'manifest': EXPECTED_MANIFEST, 'run_root': str(RUN_ROOT)}, indent=2))


In [ ]:
!pkrvision plan-classification-ablation data/processed/classification-v1/samples.jsonl artifacts/mlruns/classification/plan.json
!python -c "import json; print([(x['requested_per_class'], x['status'], x['reason']) for x in json.load(open('artifacts/mlruns/classification/plan.json'))])"


Run all feasible conditions. This cell skips any completed run with `run.json`, so rerunning a Kaggle session can continue from saved/copied outputs.

In [ ]:
from pathlib import Path
import subprocess
for level in ('10', '50', '100', 'full'):
    for augmented in (False, True):
        suffix = 'aug' if augmented else 'noaug'
        run_dir = Path(f'artifacts/mlruns/classification/n{level}-{suffix}')
        run_json = run_dir / 'run.json'
        if run_json.is_file():
            print(f'SKIP completed run: n{level}-{suffix}')
            continue
        command = ['pkrvision', 'train-classifier', 'configs/research/ablation.yaml', 'data/processed/classification-v1/internal', 'artifacts/mlruns/classification/plan.json', level, str(run_dir)]
        if augmented:
            command.append('--augmentation')
        subprocess.run(command, check=True)
!pkrvision audit-classification-runs artifacts/mlruns/classification/plan.json artifacts/mlruns/classification artifacts/mlruns/classification/run-matrix.json
!python -c "import json; print(json.load(open('artifacts/mlruns/classification/run-matrix.json'))['counts'])"


Run this after both full-data runs exist. It selects the production candidate, evaluates Abduls, exports ONNX, validates ONNX against PyTorch, and writes Grad-CAM diagnostics.

In [ ]:
!pkrvision select-classifier artifacts/mlruns/classification/nfull-noaug/run.json artifacts/mlruns/classification/nfull-aug/run.json artifacts/mlruns/classification/selection.json
with open('artifacts/mlruns/classification/selection.json') as selection_file:
    selection = json.load(selection_file)
selected_suffix = 'aug' if selection['selected'] == 'registered_augmentation' else 'noaug'
checkpoint = f'artifacts/mlruns/classification/nfull-{selected_suffix}/best.pt'
!mkdir -p artifacts/mlruns/classification/release artifacts/models/release
!pkrvision evaluate-external $checkpoint data/processed/classification-v1/external artifacts/mlruns/classification/external/result.json
!pkrvision export-classifier $checkpoint artifacts/models/release/classifier.onnx
!pkrvision validate-classifier-onnx $checkpoint artifacts/models/release/classifier.onnx data/processed/classification-v1/internal/test artifacts/models/release/onnx-validation.json
predictions = f'artifacts/mlruns/classification/nfull-{selected_suffix}/test-predictions.npz'
!pkrvision generate-gradcam $checkpoint data/processed/classification-v1/internal/test $predictions artifacts/mlruns/classification/release/gradcam
!tar -czf /kaggle/working/pkr-classification-results.tar.gz artifacts/mlruns/classification artifacts/models/release


Download `/kaggle/working/pkr-classification-results.tar.gz` from the Kaggle output panel, or save a Kaggle Notebook version so the archive is retained as an output artifact. Do not copy metrics into README/docs until the result JSON is back in the local repo and validated.